# Causal Discovery Analysis

Causal search using BOSS/FGES algorithms via PyTetrad.

**Reference:** Kumu R package, `issue_causal_analysis.Rmd` (Stage 5: Causal Search)

## 1. Configuration

Set algorithm parameters and file paths.

In [11]:
# Analysis parameters
ALGORITHM = "fges"  # Options: "boss" or "fges"
DATA_PATH = "null_variable_dt.csv"
KNOWLEDGE_FILE = "mike_knowledge_box.txt"
OUTPUT_DIR = "boss_results" if ALGORITHM == "boss" else "fges_results"

# Algorithm-specific parameters
PENALTY_DISCOUNT = 2
N_BOOTSTRAP = 100 if ALGORITHM == "boss" else 50
SEED = 32

## 2. Import Libraries

Load required modules for data processing and causal analysis.

In [12]:
import pandas as pd
from pykumu_helpers import load_data, detect_variable_types
from pykumu_algorithms import run_boss_analysis, run_fges_analysis

In [13]:
# Reload modules to pick up code changes
import importlib
import sys
if 'pykumu_algorithms' in sys.modules:
    importlib.reload(sys.modules['pykumu_algorithms'])
if 'pykumu_helpers' in sys.modules:
    importlib.reload(sys.modules['pykumu_helpers'])
print("Modules reloaded")

Modules reloaded


## 3. Load Data

Read dataset and validate data quality.

In [14]:
data = load_data(DATA_PATH)
print(f"Dataset: {data.shape[0]} rows × {data.shape[1]} columns")

Dataset: 4870 rows × 138 columns


## 4. Variable Analysis

Identify binary CVE indicators, continuous metrics, and null variables.

In [15]:
var_types = detect_variable_types(data)
print(f"Binary indicators: {len(var_types['cve_indicators'])}")
print(f"Continuous metrics: {len(var_types['metrics'])}")
print(f"Null variables: {len(var_types['null_variables'])}")

Binary indicators: 98
Continuous metrics: 17
Null variables: 23


## 5. Causal Search

Executes causal discovery with bootstrapping and domain knowledge constraints.

In [16]:
# Execute algorithm
if ALGORITHM == "boss":
    results = run_boss_analysis(
        data=data,
        knowledge_file=KNOWLEDGE_FILE,
        output_dir=OUTPUT_DIR,
        penalty_discount=PENALTY_DISCOUNT,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED
    )
else:
    results = run_fges_analysis(
        data=data,
        knowledge_file=KNOWLEDGE_FILE,
        output_dir=OUTPUT_DIR,
        penalty_discount=PENALTY_DISCOUNT,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED
    )

print(f"\nDiscovered: {results['n_nodes']} nodes, {results['n_edges']} edges")
print(f"Elapsed: {results['elapsed_time']:.1f}s")

FGES completed: 138 nodes, 105 edges
Elapsed: 7.3s

Discovered: 138 nodes, 105 edges
Elapsed: 7.3s


## 6. Results Summary

Examine discovered edges and their directionality.

In [17]:
edges_df = results['edges']
print(f"Total edges: {len(edges_df)}")
print(f"\nEdge types:")
print(f"  Directed (2→3): {len(edges_df[(edges_df['Endpoint_From']==2) & (edges_df['Endpoint_To']==3)])}")
print(f"  Directed (3→2): {len(edges_df[(edges_df['Endpoint_From']==3) & (edges_df['Endpoint_To']==2)])}")
print(f"  Undirected (3—3): {len(edges_df[(edges_df['Endpoint_From']==3) & (edges_df['Endpoint_To']==3)])}")

print(f"\nFirst 10 edges:")
edges_df.head(10)

Total edges: 105

Edge types:
  Directed (2→3): 47
  Directed (3→2): 58
  Undirected (3—3): 0

First 10 edges:


,From,To,Endpoint_From,Endpoint_To
0,b_130166,nv-b_104180,2,3
1,b_160705,start,3,2
2,b_162107,start,3,2
3,b_162180,start,3,2
4,b_173733,nv-b_102939,2,3
5,b_62940,start,2,3
6,b_63738,start,3,2
7,b_64339,start,2,3
8,b_81672,start,3,2
9,churn,churn2,3,2


## 7. Adjacency Matrix

Matrix representation of graph structure (0=none, 1=circle, 2=arrow, 3=tail).

In [18]:
adjacency = results['adjacency_matrix']
print(f"Adjacency matrix: {adjacency.shape[0]}×{adjacency.shape[1]}")
adjacency.iloc[:5, :5]

Adjacency matrix: 138×138


,b_100433,b_100740,b_100742,b_102939,b_103864
0,0,0,0,0,0
1,0,0,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,0


## 8. Graph String

Text representation of the causal graph.

In [19]:
graph_str = str(results['graph_string'])
print(f"Graph representation ({len(graph_str)} characters):")
print(graph_str[:500] + "...")

Graph representation (14132 characters):
Graph Nodes:
b_100433;b_100740;b_100742;b_102939;b_103864;b_104180;b_113207;b_114109;b_114576;b_114577;b_114619;b_120027;b_120884;b_122110;b_122333;b_130166;b_134353;b_136450;b_140076;b_140160;b_140195;b_140221;b_140224;b_142970;b_143470;b_143505;b_143506;b_143507;b_143508;b_143509;b_143510;b_143511;b_143513;b_143567;b_143568;b_143569;b_143570;b_143571;b_143572;b_148275;b_150204;b_150205;b_150206;b_150207;b_150208;b_150209;b_150285;b_150286;b_150287;b_150288;b_150289;b_150290;b_150291;b_151787;b...


## 9. JSON Representation

JSON format of the causal graph for API integration and visualization tools.

In [20]:
import json

# Get JSON representation from results (already extracted and saved)
graph_json = results['graph_json']

# Parse and pretty-print
json_data = json.loads(str(graph_json))
print(f"JSON structure with {len(json_data.get('nodes', []))} nodes and {len(json_data.get('edges', []))} edges")
print(f"\nFirst 3 nodes:")
print(json.dumps(json_data.get('nodes', [])[:3], indent=2))
print(f"\nFirst 3 edges:")
print(json.dumps(json_data.get('edges', [])[:3], indent=2))
print(f"\nFull JSON ({len(str(graph_json))} characters)")

JSON structure with 138 nodes and 0 edges

First 3 nodes:
[
  {
    "attributes": {
      "Score": 10836.555666363318
    },
    "nodeType": "MEASURED",
    "nodeVariableType": "DOMAIN",
    "centerX": -1,
    "centerY": -1,
    "rank": -1,
    "selectionBias": false,
    "name": "b_100433"
  },
  {
    "attributes": {
      "Score": 10113.370754855294
    },
    "nodeType": "MEASURED",
    "nodeVariableType": "DOMAIN",
    "centerX": -1,
    "centerY": -1,
    "rank": -1,
    "selectionBias": false,
    "name": "b_100740"
  },
  {
    "attributes": {
      "Score": 14355.792934988858
    },
    "nodeType": "MEASURED",
    "nodeVariableType": "DOMAIN",
    "centerX": -1,
    "centerY": -1,
    "rank": -1,
    "selectionBias": false,
    "name": "b_100742"
  }
]

First 3 edges:
[]

Full JSON (6009396 characters)


---

## 10. Output Files

All results have been saved to the `{OUTPUT_DIR}` directory:

**Saved files:**
- `adjacency_matrix.csv` - Full 138×138 adjacency matrix (0=none, 1=circle, 2=arrow, 3=tail)
- `edge_list.csv` - Discovered edges with directionality
- `graph_string.txt` - Text representation of the causal graph
- `graph.json` - JSON format for API integration and visualization tools

**Usage examples:**
```python
# Load adjacency matrix
adj = pd.read_csv(f"{OUTPUT_DIR}/adjacency_matrix.csv", index_col=0)

# Load edges
edges = pd.read_csv(f"{OUTPUT_DIR}/edge_list.csv")

# Load JSON for web visualization
import json
with open(f"{OUTPUT_DIR}/graph.json", "r") as f:
    graph_data = json.load(f)
```